# NOTEBOOK FEATURE ENGINEERING

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\Nitropc\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_eda.csv")

In [3]:
df.columns

Index(['sex', 'race', 'age', 'age_cat', 'decile_score', 'v_decile_score',
       'is_recid', 'is_violent_recid', 'score_text', 'screening_date',
       'two_year_recid', 'priors_count', 'c_charge_degree', 'start', 'end',
       'event', 'c_jail_in', 'c_jail_out', 'juv_fel_count', 'juv_misd_count',
       'juv_other_count', 'person_id', 'agency_text', 'maritalstatus',
       'language', 'rawscore', 'days_in_jail', 'juv_priors_count'],
      dtype='object')

In [4]:
lista_variables_modelo = [
    'person_id',
    'decile_score',
    'rawscore',
    'v_decile_score',
    'is_recid',
    'is_violent_recid',
    'two_year_recid',
    'sex',
    'race',
    'age',
    'days_in_jail',
    'priors_count',
    'juv_priors_count',
    'c_charge_degree',
    'maritalstatus'
]

In [5]:
df = df[lista_variables_modelo]

In [6]:
df.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'days_in_jail', 'priors_count', 'juv_priors_count', 'c_charge_degree',
       'maritalstatus'],
      dtype='object')

In [7]:
df['race'] = df['race'].replace({'asian': 'other', 'native american': 'other'})

In [8]:
df.race.value_counts()

race
african-american    2944
caucasian           2031
hispanic             512
other                363
Name: count, dtype: int64

In [9]:
df["two_year_recid"].sum()

np.int64(2181)

In [10]:
df.shape

(5850, 15)

In [11]:
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})

In [12]:
df.sex.value_counts()

sex
0    4719
1    1131
Name: count, dtype: int64

In [13]:
df['c_charge_degree'] = df['c_charge_degree'].replace({'felony': 0, 'misdemeanor': 1})

In [14]:
df.c_charge_degree.value_counts()

c_charge_degree
0    3731
1    2119
Name: count, dtype: int64

In [15]:
df['maritalstatus'] = df['maritalstatus'].replace({'married': 'significant other', 'divorced': 'separated', 'widowed': 'other', 'unknown': 'other'})

In [16]:
df.maritalstatus.value_counts()

maritalstatus
single               4531
significant other     891
separated             376
other                  52
Name: count, dtype: int64

In [17]:
def one_hot_encoding(df, column, drop_val):
    encoder = OneHotEncoder(
    drop=[drop_val], 
    sparse_output=False
    )

    encoded = encoder.fit_transform(df[[column]])

    df_encoded = pd.DataFrame(
    encoded,
    columns = encoder.get_feature_names_out([column])
    )

    return df_encoded

In [18]:
df_marital_status = pd.concat([df, one_hot_encoding(df, 'maritalstatus', 'single')], axis = 1)

In [19]:
df_no_caucasian = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'caucasian')], axis = 1)

In [20]:
df_no_african = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'african-american')], axis = 1)

In [21]:
df.to_csv(r'C:\Users\Nitropc\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_modelo.csv', index=False)